In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('nlp').getOrCreate()

24/07/13 02:45:38 WARN Utils: Your hostname, javad resolves to a loopback address: 127.0.1.1; using 192.168.1.39 instead (on interface wlo1)
24/07/13 02:45:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/13 02:45:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = spark.read.csv("data",inferSchema=True,sep='\t')

In [4]:
data = data.withColumnRenamed('_c0','class').withColumnRenamed('_c1','text')

In [5]:
data.show()

+-----+--------------------+
|class|                text|
+-----+--------------------+
|  ham|Go until jurong p...|
|  ham|Ok lar... Joking ...|
| spam|Free entry in 2 a...|
|  ham|U dun say so earl...|
|  ham|Nah I don't think...|
| spam|FreeMsg Hey there...|
|  ham|Even my brother i...|
|  ham|As per your reque...|
| spam|WINNER!! As a val...|
| spam|Had your mobile 1...|
|  ham|I'm gonna be home...|
| spam|SIX chances to wi...|
| spam|URGENT! You have ...|
|  ham|I've been searchi...|
|  ham|I HAVE A DATE ON ...|
| spam|XXXMobileMovieClu...|
|  ham|Oh k...i'm watchi...|
|  ham|Eh u remember how...|
|  ham|Fine if thats th...|
| spam|England v Macedon...|
+-----+--------------------+
only showing top 20 rows



In [6]:
from pyspark.sql.functions import length

In [7]:
data = data.withColumn('length',length(data['text']))

In [8]:
data.show()

+-----+--------------------+------+
|class|                text|length|
+-----+--------------------+------+
|  ham|Go until jurong p...|   111|
|  ham|Ok lar... Joking ...|    29|
| spam|Free entry in 2 a...|   155|
|  ham|U dun say so earl...|    49|
|  ham|Nah I don't think...|    61|
| spam|FreeMsg Hey there...|   147|
|  ham|Even my brother i...|    77|
|  ham|As per your reque...|   160|
| spam|WINNER!! As a val...|   157|
| spam|Had your mobile 1...|   154|
|  ham|I'm gonna be home...|   109|
| spam|SIX chances to wi...|   136|
| spam|URGENT! You have ...|   155|
|  ham|I've been searchi...|   196|
|  ham|I HAVE A DATE ON ...|    35|
| spam|XXXMobileMovieClu...|   149|
|  ham|Oh k...i'm watchi...|    26|
|  ham|Eh u remember how...|    81|
|  ham|Fine if thats th...|    56|
| spam|England v Macedon...|   155|
+-----+--------------------+------+
only showing top 20 rows



In [9]:
# Pretty Clear Difference
data.groupby('class').mean().show()

+-----+-----------------+
|class|      avg(length)|
+-----+-----------------+
|  ham|71.45431945307645|
| spam|138.6706827309237|
+-----+-----------------+



In [10]:
from pyspark.ml.feature import Tokenizer,StopWordsRemover, CountVectorizer,IDF,StringIndexer

tokenizer = Tokenizer(inputCol="text", outputCol="token_text")
stopremove = StopWordsRemover(inputCol='token_text',outputCol='stop_tokens')
count_vec = CountVectorizer(inputCol='stop_tokens',outputCol='c_vec')
idf = IDF(inputCol="c_vec", outputCol="tf_idf")
ham_spam_to_num = StringIndexer(inputCol='class',outputCol='label')

In [11]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vector

In [12]:
clean_up = VectorAssembler(inputCols=['tf_idf','length'],outputCol='features')

In [13]:
from pyspark.ml.classification import NaiveBayes

In [14]:
# Use defaults
nb = NaiveBayes()

In [15]:
from pyspark.ml import Pipeline

In [16]:
data_prep_pipe = Pipeline(stages=[ham_spam_to_num,tokenizer,stopremove,count_vec,idf,clean_up])

In [17]:
cleaner = data_prep_pipe.fit(data)

In [18]:
clean_data = cleaner.transform(data)

In [19]:
clean_data = clean_data.select(['label','features'])

In [20]:
clean_data.show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(13424,[7,11,31,6...|
|  0.0|(13424,[0,24,301,...|
|  1.0|(13424,[2,13,19,3...|
|  0.0|(13424,[0,70,80,1...|
|  0.0|(13424,[36,134,31...|
|  1.0|(13424,[10,60,140...|
|  0.0|(13424,[10,53,102...|
|  0.0|(13424,[127,185,4...|
|  1.0|(13424,[1,47,121,...|
|  1.0|(13424,[0,1,13,27...|
|  0.0|(13424,[18,43,117...|
|  1.0|(13424,[8,16,37,8...|
|  1.0|(13424,[13,30,47,...|
|  0.0|(13424,[39,95,221...|
|  0.0|(13424,[555,1797,...|
|  1.0|(13424,[30,109,11...|
|  0.0|(13424,[82,214,44...|
|  0.0|(13424,[0,2,49,13...|
|  0.0|(13424,[0,74,105,...|
|  1.0|(13424,[4,30,33,5...|
+-----+--------------------+
only showing top 20 rows



In [21]:
(training,testing) = clean_data.randomSplit([0.7,0.3])

In [22]:
spam_predictor = nb.fit(training)

24/07/13 02:49:47 WARN DAGScheduler: Broadcasting large task binary with size 1164.9 KiB
24/07/13 02:49:47 WARN DAGScheduler: Broadcasting large task binary with size 1144.2 KiB


In [23]:
data.printSchema()

root
 |-- class: string (nullable = true)
 |-- text: string (nullable = true)
 |-- length: integer (nullable = true)



In [24]:
test_results = spam_predictor.transform(testing)

In [25]:
test_results.show()

24/07/13 02:50:14 WARN DAGScheduler: Broadcasting large task binary with size 1370.3 KiB


+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(13424,[0,1,9,14,...|[-558.99278839875...|[1.0,3.0268123486...|       0.0|
|  0.0|(13424,[0,1,9,14,...|[-558.99278839875...|[1.0,3.0268123486...|       0.0|
|  0.0|(13424,[0,1,12,33...|[-440.89472606342...|[1.0,6.3579124263...|       0.0|
|  0.0|(13424,[0,1,14,18...|[-1360.0062744888...|[1.0,3.2481343399...|       0.0|
|  0.0|(13424,[0,1,14,31...|[-217.24876610832...|[1.0,1.8189527507...|       0.0|
|  0.0|(13424,[0,1,14,79...|[-707.32845971108...|[1.0,5.9327443404...|       0.0|
|  0.0|(13424,[0,1,18,20...|[-852.86045944129...|[1.0,1.8836370030...|       0.0|
|  0.0|(13424,[0,1,20,27...|[-971.40981735854...|[1.0,2.9246207418...|       0.0|
|  0.0|(13424,[0,1,27,35...|[-1488.4594716319...|[0.99985989509437...|       0.0|
|  0.0|(13424,[0

24/07/13 02:50:14 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [26]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [27]:
acc_eval = MulticlassClassificationEvaluator()
acc = acc_eval.evaluate(test_results)
print("Accuracy of model at predicting spam was: {}".format(acc))

24/07/13 02:50:32 WARN DAGScheduler: Broadcasting large task binary with size 1374.9 KiB


Accuracy of model at predicting spam was: 0.9238603144756857
